# Gym99 — 4-way Experiment: BBox Norm × LR Warmup
Runs four training instances in sequence and plots a comparison:

| Run | BBox Norm | LR Warmup |
|-----|-----------|------------|
| 1   | ✗         | ✗  (baseline) |
| 2   | ✓         | ✗ |
| 3   | ✗         | ✓ |
| 4   | ✓         | ✓ |

## 0. Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/schizocatto/Yolo-ST-GCN.git'
REPO_DIR = Path('/content/Yolo-ST-GCN')
BRANCH   = 'refactor-1'

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy', 'scipy', 'torch', 'huggingface_hub', 'scikit-learn', 'tqdm'], check=True)

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '-b', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', 'origin', BRANCH], check=True)

os.chdir(str(REPO_DIR))
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print('Repo ready:', os.getcwd())

## 1. Config — edit here only

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────
GYM288_PATH = '/content/Gym288-skeleton/gym_288_skeleton.pkl'
GYM99_PATH  = '/content/Gym99-from-Gym288/gym99_from_gym288.pkl'
OUT_DIR     = 'outputs/smoke_4way'

# ── Model ────────────────────────────────────────────────────────────────
JOINT_SPEC     = 'coco18'
USE_TWO_STREAM = True

# ── Training ─────────────────────────────────────────────────────────────
EPOCHS           = 10
BATCH_SIZE       = 256
LR               = 1e-3
WEIGHT_DECAY     = 1e-4
WARMUP_EPOCHS    = 3
LOSS_NAME        = 'focal'
FOCAL_GAMMA      = 2.0
FOCAL_ALPHA_MODE = 'sqrt_inverse'
SEED             = 42

# ── Smoke sample limits ───────────────────────────────────────────────────
# Stratified proportional sampling — heavy tail is preserved.
# Set to 0 to use the full dataset.
MAX_TRAIN = 8192
MAX_VAL   = 2048

print(f'epochs={EPOCHS}  batch={BATCH_SIZE}  lr={LR}  warmup={WARMUP_EPOCHS}')
print(f'max_train={MAX_TRAIN}  max_val={MAX_VAL}')

## 2. Download Gym288

In [ ]:
from huggingface_hub import snapshot_download

dl = Path(GYM288_PATH).parent
if not (dl.exists() and any(dl.iterdir())):
    print('Downloading Gym288-skeleton...')
    dl.mkdir(parents=True, exist_ok=True)
    snapshot_download(repo_id='Lozumi/Gym288-skeleton', repo_type='dataset',
                      local_dir=str(dl), local_dir_use_symlinks=False)
else:
    print('Gym288 already present.')

pkl_candidates = sorted(Path(GYM288_PATH).parent.rglob('*.pkl'))
GYM288_PATH = str(pkl_candidates[0])
print('Gym288 pickle:', GYM288_PATH)

## 3. Build Gym99 from Gym288

In [ ]:
from src.gym99_builder import build_gym99_from_gym288_pickle

Path(GYM99_PATH).parent.mkdir(parents=True, exist_ok=True)
stats = build_gym99_from_gym288_pickle(
    gym288_dataset_path=GYM288_PATH,
    gym99_dataset_path=GYM99_PATH,
)
print('Gym99 mapping stats:')
for k, v in stats.items():
    print(f'  {k}: {v}')

## 4. Load full data (once)

In [ ]:
import torch
import numpy as np
from src.gym99_dataset import build_gym99_data_tensors, infer_num_gym99_classes
from src.config import GYM99_NUM_CLASSES

if USE_TWO_STREAM:
    _data, _bone, _labels, _flags, _, _ = build_gym99_data_tensors(
        dataset_path=GYM99_PATH, joint_spec_name=JOINT_SPEC,
        split='all', keep_unknown_split=False, return_bone_data=True,
    )
else:
    _data, _labels, _flags, _, _ = build_gym99_data_tensors(
        dataset_path=GYM99_PATH, joint_spec_name=JOINT_SPEC,
        split='all', keep_unknown_split=False,
    )
    _bone = None

num_classes = infer_num_gym99_classes(GYM99_PATH, fallback=GYM99_NUM_CLASSES)
print(f'Loaded  data={_data.shape}  labels={_labels.shape}  num_classes={num_classes}')
print(f'Split   train={(_flags==1).sum()}  test={(_flags==0).sum()}')

## 5. Stratified sampling
Samples proportionally per class so the heavy-tail distribution is preserved in the smoke subset.

In [ ]:
from collections import Counter

def stratified_indices(labels: np.ndarray, max_samples: int, seed: int = 42) -> np.ndarray:
    if max_samples <= 0 or max_samples >= len(labels):
        return np.arange(len(labels))
    rng    = np.random.default_rng(seed)
    counts = Counter(labels.tolist())
    total  = len(labels)
    chosen = []
    for cls, cnt in counts.items():
        quota = max(1, int(cnt / total * max_samples))
        quota = min(quota, cnt)
        chosen.append(rng.choice(np.where(labels == cls)[0], size=quota, replace=False))
    result = np.concatenate(chosen)
    rng.shuffle(result)
    return result


train_mask = _flags == 1
val_mask   = _flags == 0

X_train_full = _data[train_mask];  y_train_full = _labels[train_mask]
X_val_full   = _data[val_mask];    y_val_full   = _labels[val_mask]
B_train_full = _bone[train_mask] if _bone is not None else None
B_val_full   = _bone[val_mask]   if _bone is not None else None

tr_idx = stratified_indices(y_train_full, MAX_TRAIN, seed=SEED)
va_idx = stratified_indices(y_val_full,   MAX_VAL,   seed=SEED)

X_train = X_train_full[tr_idx];  y_train = y_train_full[tr_idx]
X_val   = X_val_full[va_idx];    y_val   = y_val_full[va_idx]
B_train = B_train_full[tr_idx] if B_train_full is not None else None
B_val   = B_val_full[va_idx]   if B_val_full   is not None else None

full_counts  = sorted(Counter(y_train_full.tolist()).values(), reverse=True)
smoke_counts = sorted(Counter(y_train.tolist()).values(),      reverse=True)
print(f'Smoke train={len(X_train)}  val={len(X_val)}')
print(f'Full  class size  min={min(full_counts)}  max={max(full_counts)}')
print(f'Smoke class size  min={min(smoke_counts)}  max={max(smoke_counts)}  (tail preserved)')

## 6. Experiment runner
- `bbox_normalize` is fully vectorized (single numpy op, no Python loops)
- Training uses **preload_vram** on CUDA: all train tensors live on GPU, manual batching avoids DataLoader overhead entirely
- Val uses a standard DataLoader (smaller, less critical)

In [ ]:
import os, json
from torch.utils.data import DataLoader
from src.dataset import PennActionDataset
from src.model import Model_STGCN
from src.two_stream_stgcn import TwoStream_STGCN
from src.train import eval_epoch
from src.losses import build_classification_criterion, compute_smoothed_alpha


def bbox_normalize(tensor: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    """Vectorized per-sample bbox norm — single numpy op, no Python loops."""
    out = tensor.copy()                                   # (N, 2, T, V, 1)
    lo  = out.min(axis=(2, 3, 4), keepdims=True)         # (N, 2, 1, 1, 1)
    hi  = out.max(axis=(2, 3, 4), keepdims=True)
    return (out - lo) / (hi - lo + eps)


def run_experiment(use_bbox_norm: bool, use_lr_warmup: bool) -> dict:
    label = f"norm={'Y' if use_bbox_norm else 'N'}  warmup={'Y' if use_lr_warmup else 'N'}"
    print(f'\n{"="*55}\n  Run: {label}\n{"="*55}')

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    use_cuda = device.type == 'cuda'

    # ── Bbox norm (vectorized) ───────────────────────────────────────────
    Xtr = bbox_normalize(X_train) if use_bbox_norm else X_train
    Xva = bbox_normalize(X_val)   if use_bbox_norm else X_val
    Btr = bbox_normalize(B_train) if (use_bbox_norm and B_train is not None) else B_train
    Bva = bbox_normalize(B_val)   if (use_bbox_norm and B_val   is not None) else B_val

    # ── Preload train tensors to GPU ─────────────────────────────────────
    # Bypasses DataLoader entirely — randperm + manual slice is much faster
    # than host→device copies every batch.
    if use_cuda:
        t_joint  = torch.as_tensor(Xtr, dtype=torch.float32, device=device)
        t_labels = torch.as_tensor(y_train, dtype=torch.long,    device=device)
        t_bone   = torch.as_tensor(Btr, dtype=torch.float32, device=device) if Btr is not None else None
        print(f'  Preloaded train tensors to {device}')
    else:
        t_joint = t_labels = t_bone = None

    # Val always uses DataLoader (smaller, fine)
    val_loader = DataLoader(
        PennActionDataset(Xva, y_val, bone_data=Bva, include_bone=USE_TWO_STREAM, joint_spec_name=JOINT_SPEC),
        batch_size=BATCH_SIZE, shuffle=False, drop_last=False, num_workers=0,
        pin_memory=use_cuda,
    )
    # Fallback DataLoader for CPU mode
    if not use_cuda:
        from src.dataset import PennActionDataset as PAD
        train_loader = DataLoader(
            PAD(Xtr, y_train, bone_data=Btr, include_bone=USE_TWO_STREAM, joint_spec_name=JOINT_SPEC),
            batch_size=BATCH_SIZE, shuffle=True, drop_last=False, num_workers=0,
        )

    # ── Fresh model ──────────────────────────────────────────────────────
    torch.manual_seed(SEED)
    model = (
        TwoStream_STGCN(num_classes=num_classes, joint_spec=JOINT_SPEC)
        if USE_TWO_STREAM
        else Model_STGCN(num_classes=num_classes, joint_spec=JOINT_SPEC)
    ).to(device)

    # ── Loss ─────────────────────────────────────────────────────────────
    focal_alpha = None
    if LOSS_NAME == 'focal' and FOCAL_ALPHA_MODE != 'none':
        focal_alpha = compute_smoothed_alpha(y_train, num_classes=num_classes, mode=FOCAL_ALPHA_MODE)
    criterion = build_classification_criterion(
        loss_name=LOSS_NAME, device=device,
        focal_gamma=FOCAL_GAMMA, focal_alpha=focal_alpha,
    )

    # ── Optimizer + Scheduler ────────────────────────────────────────────
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    if use_lr_warmup:
        warmup_ep = min(WARMUP_EPOCHS, EPOCHS - 1)
        cosine_ep = EPOCHS - warmup_ep
        scheduler = torch.optim.lr_scheduler.SequentialLR(
            optimizer,
            schedulers=[
                torch.optim.lr_scheduler.LinearLR(
                    optimizer, start_factor=1e-2, end_factor=1.0, total_iters=warmup_ep),
                torch.optim.lr_scheduler.CosineAnnealingLR(
                    optimizer, T_max=max(1, cosine_ep), eta_min=0.0),
            ],
            milestones=[warmup_ep],
        )
        print(f'  Scheduler: {warmup_ep} warmup + {cosine_ep} cosine')
    else:
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=max(1, EPOCHS), eta_min=0.0)
        print(f'  Scheduler: cosine only')

    # ── Training loop ────────────────────────────────────────────────────
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [],
               'val_acc': [], 'val_f1': [], 'lr': []}
    n_samples   = len(y_train)
    n_batches   = max(1, (n_samples + BATCH_SIZE - 1) // BATCH_SIZE)

    for epoch in range(EPOCHS):
        current_lr = optimizer.param_groups[0]['lr']
        model.train()
        total_loss, total_correct, total_seen = 0.0, 0, 0

        if use_cuda:
            # ── Preloaded GPU path ──────────────────────────────────────
            perm = torch.randperm(n_samples, device=device)
            for start in range(0, n_samples, BATCH_SIZE):
                idx         = perm[start:start + BATCH_SIZE]
                jnt         = t_joint[idx]
                lbl         = t_labels[idx]
                bon         = t_bone[idx] if t_bone is not None else None

                optimizer.zero_grad()
                out  = model(jnt, bon) if bon is not None else model(jnt)
                loss = criterion(out, lbl)
                loss.backward()
                optimizer.step()

                total_loss    += loss.item()
                total_correct += (out.argmax(1) == lbl).sum().item()
                total_seen    += lbl.size(0)
        else:
            # ── CPU fallback: standard DataLoader ──────────────────────
            from src.train import train_epoch
            tr_loss, tr_acc = train_epoch(
                model, train_loader, criterion, optimizer, device, show_progress=False)
            total_loss    = tr_loss * n_batches
            total_correct = int(tr_acc * n_samples)
            total_seen    = n_samples

        tr_loss = total_loss / n_batches
        tr_acc  = total_correct / total_seen if total_seen > 0 else 0.0

        val_loss, val_acc, val_f1, _, _ = eval_epoch(
            model, val_loader, criterion, device, show_progress=False)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)
        history['lr'].append(current_lr)

        print(f'  ep {epoch+1:2d}/{EPOCHS}  lr={current_lr:.2e}  '
              f'tr_loss={tr_loss:.4f}  tr_acc={tr_acc:.4f}  '
              f'val_loss={val_loss:.4f}  val_acc={val_acc:.4f}  val_f1={val_f1:.4f}')

    history['label'] = label
    return history

## 7. Run all 4 experiments

In [ ]:
configs = [
    dict(use_bbox_norm=False, use_lr_warmup=False),  # 1. baseline
    dict(use_bbox_norm=True,  use_lr_warmup=False),  # 2. norm only
    dict(use_bbox_norm=False, use_lr_warmup=True),   # 3. warmup only
    dict(use_bbox_norm=True,  use_lr_warmup=True),   # 4. both
]

all_histories = [run_experiment(**cfg) for cfg in configs]

print('\nAll runs complete.')

## 8. Comparison plots

In [ ]:
import matplotlib.pyplot as plt

colors   = ['#4878cf', '#6acc65', '#d65f5f', '#b47cc7']
epochs_x = range(1, EPOCHS + 1)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

metrics = [
    ('train_loss', 'Train Loss'),
    ('val_loss',   'Val Loss'),
    ('train_acc',  'Train Acc'),
    ('val_acc',    'Val Acc'),
    ('val_f1',     'Val Macro-F1'),
    ('lr',         'Learning Rate'),
]

for ax, (key, title) in zip(axes, metrics):
    for h, color in zip(all_histories, colors):
        ax.plot(epochs_x, h[key], label=h['label'], color=color, linewidth=1.8)
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    if key == 'lr':
        ax.set_yscale('log')
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

fig.suptitle(f'4-way comparison  (epochs={EPOCHS}  train={len(X_train)}  val={len(X_val)})',
             fontsize=12, y=1.01)
plt.tight_layout()
os.makedirs(OUT_DIR, exist_ok=True)
fig.savefig(os.path.join(OUT_DIR, 'comparison.png'), bbox_inches='tight', dpi=120)
plt.show()
print('Saved comparison.png')

In [ ]:
# Summary table
print(f'{"Run":<35} {"tr_loss":>9} {"val_loss":>9} {"val_acc":>9} {"val_f1":>9}')
print('-' * 75)

summary = []
for h in all_histories:
    row = {
        'label':          h['label'],
        'final_tr_loss':  round(h['train_loss'][-1], 4),
        'final_val_loss': round(h['val_loss'][-1],   4),
        'final_val_acc':  round(h['val_acc'][-1],    4),
        'final_val_f1':   round(h['val_f1'][-1],     4),
    }
    summary.append(row)
    print(f"{row['label']:<35} {row['final_tr_loss']:>9.4f} {row['final_val_loss']:>9.4f} "
          f"{row['final_val_acc']:>9.4f} {row['final_val_f1']:>9.4f}")

with open(os.path.join(OUT_DIR, 'summary.json'), 'w') as f:
    json.dump({
        'config': {
            'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'lr': LR,
            'warmup_epochs': WARMUP_EPOCHS, 'max_train': MAX_TRAIN, 'max_val': MAX_VAL,
            'loss': LOSS_NAME, 'focal_alpha_mode': FOCAL_ALPHA_MODE,
        },
        'results': summary,
    }, f, indent=2)
print('\nSaved summary.json')